# Ordinal SciBERT Fine-Tuning with Metadata and QWK Optimization

This notebook trains a SciBERT-based model for ordinal paper label prediction.

Main improvements compared with a simple regression baseline:

1. Richer input text: title, venue, year, authors, and DOI.
2. Venue embedding as an additional metadata feature.
3. Ordinal classification using 4 binary heads: `P(label > 1)`, `P(label > 2)`, `P(label > 3)`, and `P(label > 4)`.
4. QWK-based validation and checkpoint selection.
5. 5-fold cross-validation and fold ensemble for final submission.
6. Automatic submission export to the `submissions/` folder.

In [1]:
# Required packages:
# pip install numpy pandas scipy scikit-learn torch transformers tqdm

import os
import re
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5
MODEL_NAME = "allenai/scibert_scivocab_uncased"

MAX_LENGTH = 160
BATCH_SIZE = 8
EPOCHS = 5

BACKBONE_LR = 1e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 0.01

LABEL_COL = "Label"

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

C:\Users\ADMIN\Desktop\data_mining_assignment\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## Load Data

The notebook assumes the following folder structure:

```text
project_root/
├── data/
│   ├── train.csv
│   ├── public_test.csv
│   └── private_test.csv
├── notebooks/
└── submissions/
```

The code automatically detects whether the notebook is run from the project root or from the notebooks/ folder.

In [2]:
def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd.parent.parent,
    ]

    for p in candidates:
        if (p / "data" / "train.csv").exists():
            return p

    raise FileNotFoundError(
        "Could not find data/train.csv. Please run this notebook from the project root "
        "or from the notebooks/ folder."
    )

ROOT = find_project_root()
DATA_DIR = ROOT / "data"
SUBMISSION_DIR = ROOT / "submissions"
MODEL_DIR = ROOT / "models"

SUBMISSION_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv")
public_test = pd.read_csv(DATA_DIR / "public_test.csv")
private_test = pd.read_csv(DATA_DIR / "private_test.csv")

if LABEL_COL not in train.columns and "label" in train.columns:
    LABEL_COL = "label"

print("Project root:", ROOT)
print("Train shape:", train.shape)
print("Public test shape:", public_test.shape)
print("Private test shape:", private_test.shape)
print("Label column:", LABEL_COL)
print(train[LABEL_COL].value_counts().sort_index())

Project root: C:\Users\ADMIN\Desktop\data_mining_assignment
Train shape: (2494, 7)
Public test shape: (298, 6)
Private test shape: (298, 6)
Label column: Label
Label
1    903
2    514
3    438
4    367
5    272
Name: count, dtype: int64


## Text and Metadata Preparation

Instead of using only the paper title, this notebook builds a richer input sequence containing:

- title
- venue
- publication year
- authors
- DOI/link

The venue is also encoded as a separate categorical feature because the publication venue is often highly informative for this task.

In [3]:
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def build_input_text(df):
    title = df["title"].map(clean_text)
    venue = df["venue"].map(clean_text)
    year = df["year"].map(clean_text)
    authors = df["authors"].map(clean_text)
    doi = df["doi"].map(clean_text)

    return (
        "Title: " + title +
        " [SEP] Venue: " + venue +
        " [SEP] Year: " + year +
        " [SEP] Authors: " + authors +
        " [SEP] DOI: " + doi
    )

train["input_text"] = build_input_text(train)
public_test["input_text"] = build_input_text(public_test)
private_test["input_text"] = build_input_text(private_test)

# Venue mapping with explicit unknown token
all_train_venues = train["venue"].fillna("UNKNOWN").astype(str).str.lower().unique().tolist()
venue2id = {"<UNK>": 0}

for v in sorted(all_train_venues):
    if v not in venue2id:
        venue2id[v] = len(venue2id)

def map_venue(x):
    if pd.isna(x):
        return 0
    return venue2id.get(str(x).lower(), 0)

train["venue_id"] = train["venue"].map(map_venue)
public_test["venue_id"] = public_test["venue"].map(map_venue)
private_test["venue_id"] = private_test["venue"].map(map_venue)

NUM_VENUES = len(venue2id)

print("Number of venues:", NUM_VENUES)
print(train[["input_text", "venue_id", LABEL_COL]].head())

Number of venues: 6
                                          input_text  venue_id  Label
0  Title: Proceedings 41st International Conferen...         2      1
1  Title: Conditionals and Temporal Conditionals ...         2      1
2  Title: Learning and Contesting Assumption-base...         2      2
3  Title: Agentified Argumentative Learning (shor...         2      1
4  Title: Empowering Public Interest Communicatio...         2      1


## QWK and Ordinal Target Utilities

The competition uses Quadratic Weighted Kappa (QWK), so model selection should be based on QWK rather than validation loss alone.

The ordinal target converts labels from 1 to 5 into four binary targets:

| Label | Ordinal target |
|-------|----------------|
|   1   |  [0, 0, 0, 0]  |
|   2   |  [1, 0, 0, 0]  |
|   3   |  [1, 1, 0, 0]  |
|   4   |  [1, 1, 1, 0]  |
|   5   |  [1, 1, 1, 1]  |

In [4]:
def quadratic_weighted_kappa(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")

def labels_to_ordinal_targets(labels):
    labels = np.asarray(labels).astype(int)
    targets = np.zeros((len(labels), 4), dtype=np.float32)

    for i, label in enumerate(labels):
        targets[i, :label - 1] = 1.0

    return targets

def ordinal_logits_to_continuous(logits):
    probs = torch.sigmoid(logits)
    return 1.0 + probs.sum(dim=1)

def apply_thresholds(pred, thresholds):
    return np.digitize(pred, thresholds) + 1

def optimize_thresholds(y_true, pred, initial=None):
    if initial is None:
        initial = np.array([1.5, 2.5, 3.5, 4.5])

    def loss(thresholds):
        thresholds = np.sort(thresholds)
        y_hat = apply_thresholds(pred, thresholds)
        return -quadratic_weighted_kappa(y_true, y_hat)

    result = minimize(
        loss,
        x0=initial,
        method="Nelder-Mead",
        options={"maxiter": 2000, "xatol": 1e-6, "fatol": 1e-6},
    )

    thresholds = np.sort(result.x)
    score = -result.fun
    return thresholds, score

## Dataset Class

The dataset returns:

- tokenized SciBERT inputs
- venue ID
- ordinal targets during training
- original label for validation

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PaperDataset(Dataset):
    def __init__(self, df, labels=None, max_length=160):
        self.texts = df["input_text"].values
        self.venue_ids = df["venue_id"].values.astype(np.int64)
        self.labels = labels
        self.max_length = max_length

        if labels is not None:
            self.ordinal_targets = labels_to_ordinal_targets(labels)
        else:
            self.ordinal_targets = None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "venue_id": torch.tensor(self.venue_ids[idx], dtype=torch.long),
        }

        if self.labels is not None:
            item["ordinal_target"] = torch.tensor(self.ordinal_targets[idx], dtype=torch.float)
            item["label"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

## Model Architecture

The model uses:

1. SciBERT encoder.
2. Mean pooling and max pooling over token embeddings.
3. Venue embedding.
4. Multi-sample dropout.
5. Four ordinal output heads.

Only the last SciBERT layers are unfrozen to reduce overfitting and training cost.

In [6]:
class OrdinalSciBERTModel(nn.Module):
    def __init__(
        self,
        model_name,
        num_venues,
        venue_dim=32,
        dropout=0.2,
        n_dropout_samples=4,
        unfreeze_last_n_layers=2,
    ):
        super().__init__()

        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size

        # Freeze all SciBERT parameters first
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Unfreeze last N encoder layers
        if hasattr(self.backbone, "encoder"):
            total_layers = len(self.backbone.encoder.layer)
            for layer_idx in range(total_layers - unfreeze_last_n_layers, total_layers):
                for param in self.backbone.encoder.layer[layer_idx].parameters():
                    param.requires_grad = True

        # Unfreeze pooler if available
        if hasattr(self.backbone, "pooler") and self.backbone.pooler is not None:
            for param in self.backbone.pooler.parameters():
                param.requires_grad = True

        self.venue_embedding = nn.Embedding(num_venues, venue_dim)

        feature_dim = hidden_size * 2 + venue_dim

        self.dropouts = nn.ModuleList([
            nn.Dropout(dropout) for _ in range(n_dropout_samples)
        ])

        self.head = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(dropout),
            nn.Linear(256, 4),
        )

    def mean_max_pooling(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()

        mean_pool = (last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

        masked_hidden = last_hidden_state.masked_fill(mask == 0, -1e9)
        max_pool = masked_hidden.max(dim=1).values

        return torch.cat([mean_pool, max_pool], dim=1)

    def forward(self, input_ids, attention_mask, venue_id):
        output = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        pooled = self.mean_max_pooling(output.last_hidden_state, attention_mask)
        venue_emb = self.venue_embedding(venue_id)

        features = torch.cat([pooled, venue_emb], dim=1)

        logits = 0
        for dropout in self.dropouts:
            logits = logits + self.head(dropout(features))

        logits = logits / len(self.dropouts)

        return logits

## Training and Evaluation Functions

The training objective is binary cross-entropy over ordinal targets.

Validation uses QWK after threshold optimization. The best checkpoint for each fold is selected by validation QWK, not validation loss.

In [7]:
def train_one_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0.0

    for batch in tqdm(loader, leave=False):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        venue_id = batch["venue_id"].to(device)
        ordinal_target = batch["ordinal_target"].to(device)

        logits = model(input_ids, attention_mask, venue_id)
        loss = criterion(logits, ordinal_target)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * input_ids.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict_continuous(model, loader):
    model.eval()
    preds = []

    for batch in tqdm(loader, leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        venue_id = batch["venue_id"].to(device)

        logits = model(input_ids, attention_mask, venue_id)
        continuous = ordinal_logits_to_continuous(logits)

        preds.append(continuous.cpu().numpy())

    return np.concatenate(preds)


def build_optimizer(model):
    backbone_params = []
    head_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        if name.startswith("backbone"):
            backbone_params.append(param)
        else:
            head_params.append(param)

    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": BACKBONE_LR},
            {"params": head_params, "lr": HEAD_LR},
        ],
        weight_decay=WEIGHT_DECAY,
    )

    return optimizer

## 5-Fold Cross-Validation Training

For each fold:

1. Train the model on 4 folds.
2. Validate on the remaining fold.
3. Optimize QWK thresholds on validation predictions.
4. Save the best model checkpoint based on validation QWK.
5. Store out-of-fold predictions for final validation analysis.

In [8]:
y = train[LABEL_COL].astype(int).values

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

oof_pred = np.zeros(len(train), dtype=np.float32)
fold_thresholds = []
fold_scores = []
model_paths = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y), 1):
    print(f"\n========== Fold {fold}/{N_SPLITS} ==========")

    train_df = train.iloc[tr_idx].reset_index(drop=True)
    valid_df = train.iloc[va_idx].reset_index(drop=True)

    y_train = y[tr_idx]
    y_valid = y[va_idx]

    train_dataset = PaperDataset(train_df, labels=y_train, max_length=MAX_LENGTH)
    valid_dataset = PaperDataset(valid_df, labels=y_valid, max_length=MAX_LENGTH)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    model = OrdinalSciBERTModel(
        model_name=MODEL_NAME,
        num_venues=NUM_VENUES,
        venue_dim=32,
        dropout=0.25,
        n_dropout_samples=4,
        unfreeze_last_n_layers=2,
    ).to(device)

    optimizer = build_optimizer(model)

    total_steps = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * 0.1)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    criterion = nn.BCEWithLogitsLoss()

    best_qwk = -1
    best_thresholds = None
    best_path = MODEL_DIR / f"ordinal_scibert_fold_{fold}.pt"

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            criterion,
        )

        valid_pred = predict_continuous(model, valid_loader)
        thresholds, valid_qwk = optimize_thresholds(y_valid, valid_pred)

        print(
            f"Fold {fold} | Epoch {epoch} | "
            f"Train Loss: {train_loss:.5f} | "
            f"Valid QWK: {valid_qwk:.6f} | "
            f"Thresholds: {thresholds}"
        )

        if valid_qwk > best_qwk:
            best_qwk = valid_qwk
            best_thresholds = thresholds
            torch.save(model.state_dict(), best_path)

    print(f"Best Fold {fold} QWK: {best_qwk:.6f}")
    print(f"Best Fold {fold} thresholds: {best_thresholds}")

    # Load best fold model and generate OOF predictions
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.to(device)

    best_valid_pred = predict_continuous(model, valid_loader)
    oof_pred[va_idx] = best_valid_pred

    fold_thresholds.append(best_thresholds)
    fold_scores.append(best_qwk)
    model_paths.append(best_path)

    del model
    torch.cuda.empty_cache()

print("\n========== Cross-Validation Summary ==========")
print("Fold QWK scores:", fold_scores)
print("Mean fold QWK:", np.mean(fold_scores))
print("Std fold QWK:", np.std(fold_scores))

oof_thresholds, oof_qwk = optimize_thresholds(y, oof_pred)
oof_labels = apply_thresholds(oof_pred, oof_thresholds)

print("\nOOF Optimized QWK:", oof_qwk)
print("OOF Thresholds:", oof_thresholds)
print("OOF Prediction Distribution:")
print(pd.Series(oof_labels).value_counts().sort_index())


========== Fold 1/5 ==========


Loading weights: 100%|████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 15326.79it/s]
[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical

Fold 1 | Epoch 1 | Train Loss: 0.57436 | Valid QWK: 0.211014 | Thresholds: [1.5202692  2.52376938 3.57581348 4.36136456]


Fold 1 | Epoch 2 | Train Loss: 0.52482 | Valid QWK: 0.445741 | Thresholds: [1.66249845 2.71810084 3.02744049 4.56286167]


Fold 1 | Epoch 3 | Train Loss: 0.46763 | Valid QWK: 0.543058 | Thresholds: [1.69014651 2.76987231 3.39971864 4.23373251]


...

Fold 5 | Epoch 2 | Train Loss: 0.52639 | Valid QWK: 0.509737 | Thresholds: [1.8392627  2.52830609 2.94150258 4.65948239]


Fold 5 | Epoch 3 | Train Loss: 0.47516 | Valid QWK: 0.602105 | Thresholds: [1.73436134 2.62935333 3.29159346 3.85212454]


Fold 5 | Epoch 4 | Train Loss: 0.45260 | Valid QWK: 0.615496 | Thresholds: [1.71557796 2.70562382 3.66157115 4.14584111]


Fold 5 | Epoch 5 | Train Loss: 0.43492 | Valid QWK: 0.621875 | Thresholds: [1.69133966 2.39665717 3.51207587 4.08432751]
Best Fold 5 QWK: 0.621875
Best Fold 5 thresholds: [1.69133966 2.39665717 3.51207587 4.08432751]



========== Cross-Validation Summary ==========
Fold QWK scores: [np.float64(0.5820148170209598), np.float64(0.5740024162234695), np.float64(0.5885041134492486), np.float64(0.5576736571963695), np.float64(0.6218745793213878)]
Mean fold QWK: 0.5848139166422871
Std fold QWK: 0.021205789084474652

OOF Optimized QWK: 0.5659987046458731
OOF Thresholds: [1.6555797  2.53282693 3.43489854 4.06694348]
OOF Prediction Distribution:
1    571
2    817
3    561
4    318
5    227
Name: count, dtype: int64


## Save OOF Predictions

Out-of-fold predictions are useful for error analysis and future ensemble experiments.
```text
submissions/oof_ordinal_scibert.csv

In [9]:
oof_df = pd.DataFrame({
    "id": train["id"].values,
    "y_true": y,
    "pred_continuous": oof_pred,
    "pred_label": apply_thresholds(oof_pred, oof_thresholds),
})

oof_path = SUBMISSION_DIR / "oof_ordinal_scibert.csv"
oof_df.to_csv(oof_path, index=False)

print("Saved OOF predictions to:", oof_path)
oof_df.head()

Saved OOF predictions to: C:\Users\ADMIN\Desktop\data_mining_assignment\submissions\oof_ordinal_scibert.csv


,id,y_true,pred_continuous,pred_label
0,0,1,1.632622,1
1,1,1,3.434314,3
2,2,2,3.230051,3
3,3,1,3.656266,4
4,4,1,1.982383,2


## Test Inference and Submission

The final prediction is the average of all fold models.  
The optimized OOF thresholds are used to convert continuous ordinal scores into integer labels from 1 to 5.

The final competition file is saved as:

```text
submissions/submission.csv

In [10]:
public_dataset = PaperDataset(public_test, labels=None, max_length=MAX_LENGTH)
private_dataset = PaperDataset(private_test, labels=None, max_length=MAX_LENGTH)

public_loader = DataLoader(
    public_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

private_loader = DataLoader(
    private_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

public_pred = np.zeros(len(public_test), dtype=np.float32)
private_pred = np.zeros(len(private_test), dtype=np.float32)

for fold, model_path in enumerate(model_paths, 1):
    print(f"Inference with fold {fold} model:", model_path)

    model = OrdinalSciBERTModel(
        model_name=MODEL_NAME,
        num_venues=NUM_VENUES,
        venue_dim=32,
        dropout=0.25,
        n_dropout_samples=4,
        unfreeze_last_n_layers=2,
    ).to(device)

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)

    public_pred += predict_continuous(model, public_loader) / len(model_paths)
    private_pred += predict_continuous(model, private_loader) / len(model_paths)

    del model
    torch.cuda.empty_cache()

public_labels = apply_thresholds(public_pred, oof_thresholds)
private_labels = apply_thresholds(private_pred, oof_thresholds)

public_submission = pd.DataFrame({
    "id": public_test["id"].values,
    "Label": public_labels.astype(int),
})

private_submission = pd.DataFrame({
    "id": private_test["id"].values,
    "Label": private_labels.astype(int),
})

submission = pd.concat(
    [public_submission, private_submission],
    axis=0,
    ignore_index=True,
)

submission_path = SUBMISSION_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)
print("Submission shape:", submission.shape)
print("Submission label distribution:")
print(submission["Label"].value_counts().sort_index())

submission.head()

Inference with fold 1 model: C:\Users\ADMIN\Desktop\data_mining_assignment\models\ordinal_scibert_fold_1.pt


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 13831.12it/s]
[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/ar

Inference with fold 2 model: C:\Users\ADMIN\Desktop\data_mining_assignment\models\ordinal_scibert_fold_2.pt


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 23261.43it/s]
[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/ar

Inference with fold 3 model: C:\Users\ADMIN\Desktop\data_mining_assignment\models\ordinal_scibert_fold_3.pt


...

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 8365.65it/s]
[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/ar

Inference with fold 5 model: C:\Users\ADMIN\Desktop\data_mining_assignment\models\ordinal_scibert_fold_5.pt


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 20646.77it/s]
[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/ar

Saved submission to: C:\Users\ADMIN\Desktop\data_mining_assignment\submissions\submission.csv
Submission shape: (596, 2)
Submission label distribution:
Label
1    166
2    224
3    123
4     47
5     36
Name: count, dtype: int64


,id,Label
0,979,5
1,1106,3
2,1894,2
3,1718,2
4,1989,2
